In [14]:
import sisl
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import ipywidgets as ipw
from utils.structure import infer_centersize
from utils import load_datastructure, hamiltonian
from utils.structure import find_nearest_atoms
from utils.plots import plot_with_center, mark_electrode # plotting
from utils.energies import hamiltonian

from fractions import Fraction


%reload_ext line_profiler




<img src="figures/flake_size_conv.svg">

In [15]:
!tree results/ -L 1

results/
├── L1_W13_C6
├── L1_W17_C8
├── L1_W21_C10
├── L1_W25_C12
├── L1_W29_C14
├── L1_W33_C16
├── L1_W5_C2
├── L1_W9_C4
├── L2_W13_C3
├── L2_W17_C4
├── L2_W21_C5
├── L2_W25_C6
├── L2_W29_C7
├── L2_W33_C8
├── L2_W5_C1
├── L2_W9_C2
└── generate_structure.py

16 directories, 1 file


In [16]:
LWC = (2,9,2)
electrode, ribbon, device, DATA = load_datastructure(LWC)
energies, ldos, lr_idx = DATA["energies"], DATA["ldos"], DATA["lr_idx"]
Nk, NE, Na = ldos.shape

atoms_style = mark_electrode(lr_idx)
# print(atoms_style)
plot_with_center(device, atoms_style=atoms_style)


params : (L, W, C) = (2, 9, 2)


AttributeError: 'Geometry' object has no attribute 'plot'

In [12]:
from sisl.physics.self_energy import RecursiveSI
from utils import lr_energies, add_lr_energies
from utils.energies import diagonal_of_inverse


def make_scaling_matrix(ldos0):
    """Create the matrix for E=0 
    used for modulating/scaling the self-energies for a range of energies

    Parameters
    ----------
    ldos0 : np.ndarray
        A array of LDOS for E=0, shape = (Na,) for Na atoms.
    """
    D = np.sqrt(ldos0) # shape (Na)
    return D[:, None] * D[None, :] # shape (Na, Na)
    
def mod_LDOS(device: 'sisl.Geometry',
               electrode: 'sisl.Geometry',
               lr_indices: tuple[np.ndarray, np.ndarray],
               *,
               energies: np.ndarray | list = [0.0],
               Nk: int = 1,
               eta: float = 1e-5,
               form: str = "csc",
               modulate_SE: bool = False) -> np.ndarray:
    energies = np.asarray(energies)
    
    C = 0.05 # scaling for Sigma = i*LDOS(E=0)*C
    Ne = len(energies) # number of energies
    k_direction = [1, Nk, 1] # direction to sample k 
    H_D = hamiltonian(device)
    Na = len(device) # number of atoms
    H_D.set_nsc([1,1,1])
    kpts = sisl.MonkhorstPack(H_D, k_direction).k
    
    H_0 = hamiltonian(electrode)
    SE = RecursiveSI(H_0, infinite="+A")
    
    # Precompute LDOS(E=0) if needed
    if modulate_SE:
        print(f"using modulated with {C=}")
        LDOS_E0 = np.zeros((Nk, Na), dtype=complex)
        E0 = 0.0 + 1j*eta
        
        for ik, kvec, in enumerate(tqdm(kpts, desc="LDOS E=0")):
            Hk = H_D.Hk(k=kvec, format=form, dtype=complex)
            Sk = H_D.Sk(k=kvec, format=form, dtype=complex)
            
            SE_pair = lr_energies(electrode=SE, En=E0, kvec=kvec)
            Hk_lr = add_lr_energies(Hk.copy(), SE_pair, lr_indices)
            
            invG = Sk*E0 - Hk_lr
            diagG = diagonal_of_inverse(invG)
            LDOS_E0[ik, :] = -np.imag(diagG) / np.pi
    
    # Compute all LDSO (optionally modulated)
    all_ldos = np.zeros(shape=(Nk, Ne, Na), dtype=float)
    for ik, kvec in enumerate(kpts):
        Hk = H_D.Hk(k=kvec, format=form, dtype=complex)
        Sk = H_D.Sk(k=kvec, format=form, dtype=complex)
        
        if modulate_SE:
            scale = make_scaling_matrix(LDOS_E0[ik, :])*C
            
        for ie, E in enumerate(tqdm(energies, desc=f"Energy for k={ik}")):
            En = E + 1j*eta
            
            SigmaL, SigmaR = lr_energies(electrode=SE, En=En, kvec=kvec)
            if modulate_SE:
                NL, _ = SigmaL.shape
                NR, _ = SigmaR.shape
                SigmaL = SigmaL + 1j*scale[:NL, :NL]
                SigmaR = SigmaR + 1j*scale[-NR:, -NR:]
            
            Hk_lr = add_lr_energies(Hk.copy(), (SigmaL, SigmaR), lr_indices)
            invG  = Sk*En - Hk_lr
            diagG = diagonal_of_inverse(invG)
            all_ldos[ik, ie, :] = -np.imag(diagG) / np.pi
            
    
    return all_ldos
        
        
        

In [13]:
mod_ldos = mod_LDOS(device=device, electrode=electrode, lr_indices=lr_idx,
         energies=energies,
         Nk=1,
         eta=1e-5,
         form="csc",
         modulate_SE=True)

using modulated with C=0.05


warn:0: SislWarning:

Geometry.close_sc has been passed an 'atoms' argument together with an R value larger than the orbital ranges. If used together with 'sparse-matrix.construct' this can result in wrong couplings.



LDOS E=0:   0%|          | 0/1 [00:00<?, ?it/s]

Energy for k=0:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
def interactive_params(device, Nk):
    # Determine k-point options
    DEVICE_CENTER_ATOMS = find_nearest_atoms(device.xyz, device.center(), neighbours=6)
    kpts = sisl.MonkhorstPack(hamiltonian(device), [Nk, 1, 1])
    options = [(f"{[str(Fraction(val).limit_denominator(len(kpts)*2)) for val in kvec]})", i) for i, kvec in enumerate(kpts)]
    ks = ipw.Dropdown(options=options, description="k")
    
    # Determine site (only the centermost)
    sites = ipw.Dropdown(options=DEVICE_CENTER_ATOMS, description="Site")
    
    # Option for using normalized data
    norm = ipw.Checkbox(value=False, description="Normalize", indent=True)
    # same = ipywidgets.Checkbox(value=True, description="Plot in same figure", indent=True)
    return ks, sites, norm


In [ ]:
def plot_ldos(ldos1, kidx, s, ldos2=None, norm=True):
    """Plot LDOS from self-energy calculation."""
    
    
    vmin = ldos1[kidx, :, sites.options].mean()
    if not ldos2 is None:
        vmin2 = ldos2[kidx, :, sites.options].mean()
        vmin = np.mean([vmin, vmin2])
    
    
    X1 = ldos1[kidx, :, s]
    
    fig, axes = plt.subplots(1,1)
    if norm: # plot in the save figure
        X1 = normalize(X1, vmin=vmin, vmax = 1)
        axes.set_xlabel("LDOS (normalized)")
    else:
        axes.set_xlabel(f"LDOS")
        
        
    
    axes.plot(X1, energies, label="mod LDOS", linestyle=":", lw=3)
    
    if not ldos2 is None:
        X2 = ldos2[kidx, :, s]
        if norm:
            X2 = normalize(X2, vmin=vmin, vmax=1)
        axes.plot(X2, energies, label="LDOS", linestyle="--", alpha=0.8)
        axes.legend()
            
    axes.set_ylabel("E")
    axes.axvline(0, color="k", linestyle="--")
    
    fig.suptitle("L={:}, W={:}, C={:}".format(*LWC))
    fig.savefig("ldos_and_mod_ldos_scale0.05.png")

In [ ]:
ks, sites, norm = interactive_params(device, Nk=Nk)
mod_ldos_fixed = ipw.fixed(mod_ldos)
ldos_fixed = ipw.fixed(ldos)

ipw.interact(plot_ldos, kidx=ks, s=sites, norm=norm, ldos1=mod_ldos_fixed, ldos2=ldos_fixed)